# Dataset exploration, preprocessed datasets via SpaCy and train-test splits

Imports

In [ ]:
import json
import pandas as pd
import matplotlib.pyplot as plt
from collections import Counter
import plotly.graph_objects as go
import numpy as np
import random 
import os

In [ ]:
import spacy
import html
import re
from nltk.stem import PorterStemmer

Number of articles, histogram of wordcount per concatenated block of text per article, article count per year, list of tags and number of articles per tag

Article count per year

In [ ]:
file_path = "raw_data/guardian_launder_1999_2026.json"
f = open(file_path, "r", encoding="utf-8")
articles = json.load(f)
f.close()  

#Checking how many articles we got in total
total = len(articles)
print("Total articles: " + str(total))

#each article is a dictionary with date field
#collecting dates in list
all_dates = []
for a in articles:
    date = a["date"]
    all_dates.append(date)

#converting list of dates to datetime objects abd extract year
all_dates_converted = pd.to_datetime(all_dates)
all_years = []
for d in all_dates_converted:
    y = d.year
    all_years.append(y)

#how many articles per year 
year_counts_dict = {}
for y in all_years:
    if y in year_counts_dict:
        year_counts_dict[y] = year_counts_dict[y] + 1
    else:
        year_counts_dict[y] = 1


all_the_years_unsorted = list(year_counts_dict.keys())
all_the_years_sorted = sorted(all_the_years_unsorted)

#sorting years
sorted_years= []
sorted_counts = []
for y in all_the_years_sorted:
    sorted_years.append(y)
    count_for_this_year = year_counts_dict[y]  
    sorted_counts.append(count_for_this_year)

#Bar chart
fig, ax = plt.subplots(figsize=(12, 4))
ax.bar(sorted_years, sorted_counts, color="steelblue", edgecolor="white", linewidth=0.4)
ax.set_xlabel("Year")
ax.set_ylabel("Number of articles")
total_formatted = f"{total:,}"  # the comma makes it easier to read big numbers
ax.set_title("Guardian AML corpus — articles per year (n=" + total_formatted + ")")
ax.set_xticks(sorted_years)
ax.tick_params(axis="x", rotation=45)
plt.tight_layout()

save_path = "processed_data/articles_per_year.png"
plt.savefig(save_path, dpi=150)
plt.show()

Tag distribution over raw dataset

In [ ]:

#tags field splitting and cleaning
all_tags_list = []
for article in articles:
    tags_raw = article["tags_keywords"]
    tags_split = tags_raw.split("|")
    tags_cleaned = []
    for t in tags_split:
        t_stripped = t.strip()
        if t_stripped != "":
            tags_cleaned.append(t_stripped)
    
    #adding cleaned tags to list
    for tag in tags_cleaned:
        all_tags_list.append(tag)

#counting tags
tag_counts_dict = {}
for tag in all_tags_list:
    #if the tag is already in the dictionary, add 1 to its count
    if tag in tag_counts_dict:
        tag_counts_dict[tag] = tag_counts_dict[tag] + 1
    else:
        tag_counts_dict[tag] = 1

#now i need to find the top 30 most common tags
#i'll do this by looping 30 times and each time finding the tag with the highest count
#then removing it so the next loop finds the next highest and so on

top_30_tags = []
top_30_counts = []
tag_counts_copy = dict(tag_counts_dict)

#repeat 30 times to get the top 30 tags
for i in range(30):
    highest_count_so_far = 0
    tag_with_highest_count = ""
    for tag in tag_counts_copy:
        count = tag_counts_copy[tag]
        if count > highest_count_so_far:
            highest_count_so_far = count
            tag_with_highest_count = tag
    
    #add the highest to list
    top_30_tags.append(tag_with_highest_count)
    top_30_counts.append(highest_count_so_far)
    
    #remove it from the copy for the next highest
    del tag_counts_copy[tag_with_highest_count]

#two columns
tag_df = pd.DataFrame()
tag_df["Tag"] = top_30_tags
tag_df["Count"] = top_30_counts

#Ploty table
fig = go.Figure(data=[go.Table(
    columnwidth=[400, 100],
    header=dict(
        values=["<b>Tag</b>", "<b>Count</b>"],  #<b> makes the text bold in plotly
        fill_color="#2c3e50",  #dark blue-ish background for the header
        font=dict(color="white", size=13),  #white text so it shows up on dark background
        align="left",
        height=36,  
    ),
    cells=dict(
        values=[tag_df["Tag"], tag_df["Count"]],  #the actual data goes here
        fill_color=[["#f4f6f9" if i % 2 == 0 else "white" for i in range(len(tag_df))]],
        font=dict(color="#2c3e50", size=12),
        align="left",
        height=28,  
    )
)])

number_of_rows = len(tag_df)
height_of_each_row = 30
extra_for_header = 40
total_height = number_of_rows * height_of_each_row + extra_for_header

fig.update_layout(
    margin=dict(l=0, r=0, t=0, b=0),  #l=left r=right t=top b=bottom all set to 0
    height=total_height,
)

save_path = "processed_data/tag_counts.png"
fig.write_image(save_path, scale=2)
fig.show()

Table with wordcount per article

In [ ]:
word_counts = []
for article in articles:
    body_text = article["bodyText"]
    #splitting on spaces
    list_of_words = body_text.split()
    number_of_words = len(list_of_words)
    word_counts.append(number_of_words)

bin_boundaries = [0, 500, 1000, 1500, 2000, 2500, 3000, 3500, 4000, 4500, 5000, np.inf]

bin_labels = []
for i in range(len(bin_boundaries) - 1):
    lower = bin_boundaries[i]
    upper = bin_boundaries[i + 1]

    if upper == np.inf:
        bin_labels.append(">5,000")
    else:
        #convert to int first so there's no decimal point in the label
        lower_int = int(lower)
        upper_int = int(upper)
        #format with commas 
        lower_str = f"{lower_int:,}"
        upper_str = f"{upper_int:,}"
        label = lower_str + "–" + upper_str
        bin_labels.append(label)

#looping over bins

bin_counts = []
for i in range(len(bin_boundaries) - 1):
    lower = bin_boundaries[i]
    upper = bin_boundaries[i + 1]
    count_for_this_bin = 0
    for wc in word_counts:
        #check if the word count is in tihs bin
        #lower boundary is included
        if wc >= lower and wc < upper:
            count_for_this_bin = count_for_this_bin + 1 
    bin_counts.append(count_for_this_bin)

#percentage in bin
total_number_of_articles = len(word_counts)

bin_percentages = []
for c in bin_counts:
    percentage = c / total_number_of_articles * 100
    percentage_rounded = round(percentage, 1)
    percentage_str = str(percentage_rounded) + "%"
    bin_percentages.append(percentage_str)

bin_counts_formatted = []
for c in bin_counts:
    count_str = f"{c:,}"
    bin_counts_formatted.append(count_str)


fig = go.Figure(data=[go.Table(
    columnwidth=[200, 100, 100],
    header=dict(
        values=["<b>Word count range</b>", "<b>Articles</b>", "<b>Share</b>"],
        fill_color="#2c3e50",  #dark background for header
        font=dict(color="white", size=13),  #white text on dark background
        align="left",
        height=36),
    cells=dict(
        values=[bin_labels, bin_counts_formatted, bin_percentages],
        #alternating row colors to make it easier to read
        fill_color=[["#f4f6f9" if i % 2 == 0 else "white" for i in range(len(bin_labels))]],
        font=dict(color="#2c3e50", size=12),
        align="left",
        height=28))])

#calculate height of the table
number_of_rows = len(bin_labels)
height_per_row = 30
header_height = 40
total_height = number_of_rows * height_per_row + header_height

fig.update_layout(
    margin=dict(l=0, r=0, t=0, b=0),  
    height=total_height)

save_path = "processed_data/wordcount_bins.png"
fig.write_image(save_path, scale=2) 
fig.show()

Inspecting long articles

In [ ]:
#find all the articles that have more than 5000 words
long_articles = []
for a in articles:
    body_text = a["bodyText"]
    #split into words and count
    list_of_words = body_text.split()
    number_of_words = len(list_of_words)
    if number_of_words > 5000:
        long_articles.append(a)

print("number of articles with more than 5000 words: " + str(len(long_articles)))

number_of_articles_to_sample = 10
sample = random.sample(long_articles, 10)

for a in sample:
    #count the words again for this article
    body_text = a["bodyText"]
    list_of_words = body_text.split()
    number_of_words = len(list_of_words)
    article_id = a["id"]
    print(str(number_of_words) + " words — " + article_id)

Removing articles with /live/ or /blog/live/ after /section/ in ID from the dataset

In [ ]:

def is_live(article_id):
    if "/live/" in article_id:
        return True
    if "/blog/live/" in article_id:
        return True
    return False

#Looping and only keeping desired article types
articles_clean = []
for a in articles:
    article_id = a["id"]
    result = is_live(article_id)
    if result == False:
        articles_clean.append(a)
number_before = len(articles)
number_after = len(articles_clean)
number_removed = number_before - number_after

print("Before: " + str(number_before))
print("After: " + str(number_after))
print("Removed: " + str(number_removed))

Updating processed articles file with removed articles

In [ ]:
save_path = "processed_data/guardian_launder_processed.json"
os.makedirs("processed_data", exist_ok=True)
output_file = open(save_path, "w", encoding="utf-8")
json.dump(articles_clean, output_file, ensure_ascii=False, indent=2)
output_file.close()  
print("Saved to " + save_path)

# SpaCy implementation and verification

Loading and importing model - tokenization, lemmatization, POS tagging and dropping undesirable tokens (integrated stopword list):
DET are articles and determiners (a, the, this, those)
AUX are auxiliary verbs (is, has, was, will, would)
PART are particles and possessive 's (to [infinitive], not, 's)
PUNCT are punctuation marks (. , ; : ! ?)
SYM are symbols ($, %, &, €)
X are unclassifiable tokens (catches stray HTML residue)
SPACE are White-space tokens

The output should produce a lemmatized and non-lemmatized version at this stage, permutated with 'nouns' only and 'all parts of speech', yielding 4 datasets for benchmarking down the line.
A fifth dataset is added for the stemmed version of the full set


In [ ]:
#loading the spacy english model - disabling redundant parts
nlp = spacy.load("en_core_web_sm", disable=["parser", "ner", "senter"])
dropped_upos = {"DET", "AUX", "PART", "PUNCT", "SYM", "X", "SPACE", "ADV"}
#repeat path to be sure
save_path = "processed_data/guardian_launder_processed.json"
#nouns only version
nouns_only = {"NOUN", "PROPN"}

#version with all words without adverbers
content_pos = {"NOUN", "PROPN", "VERB", "ADJ"}

#native stopword list
stopwords = nlp.Defaults.stop_words

#build a dictionary that maps article id to its date
date_map = {}
for a in articles_clean:
    article_id = a["id"]
    article_date = a["date"]
    date_map[article_id] = article_date

#set up the porter stemmer
_porter = PorterStemmer()

#additional stemming rules
pre_stem_rules = {
    "ial": "",
    "ier": "",
}

def porter_stem(word):
    for suffix, replacement in pre_stem_rules.items():
        if word.endswith(suffix):
            word = word[:-len(suffix)] + replacement
            #stop after the first match
            break
    #now apply the porter stemmer
    stemmed = _porter.stem(word)
    return stemmed

#removing numbers, punctuation, and extra spaces
def clean_text(t):
    #html tags
    t = html.unescape(t)
    #remove numbers and words that start with numbers 
    t = re.sub(r'\b\d+[\w]*\b', '', t)
    #white space removal should not affect hyphenated words
    t = re.sub(r'\s-\s', '-', t)
    #remove all punctuation
    t = re.sub(r'[^\w\s]', '', t)
    #remove single character words like "a" or "i"
    t = re.sub(r'\b\w\b', '', t)
    #replace multiple spaces with a single space and remove leading/trailing spaces
    t = re.sub(r'\s+', ' ', t).strip()
    return t

#concatening articles to process
all_texts_to_process = []
for a in articles_clean:
    title = a.get("title", "") or ""
    trail = a.get("trailText", "") or ""
    body = a.get("bodyText", "") or ""
    combined_text = title + " " + trail + " " + body
    all_texts_to_process.append(combined_text)

processed_docs = list(nlp.pipe(all_texts_to_process, batch_size=64))

#token extraction
processed = []
for i in range(len(processed_docs)):
    doc = processed_docs[i]
    article = articles_clean[i]
    
    #get the article id
    article_id = article.get("id")
    
    #version 1: all POS minus dropped, not lemmatized 
    tokens_all = []
    for token in doc:
        #skip tokens with dropped POS tags
        if token.pos_ in dropped_upos:
            continue
        #skip stopwords
        if token.text.lower() in stopwords:
            continue
        #skip whitespace tokens
        if token.is_space:
            continue
        tokens_all.append(token.text.lower())
    
    #version 2: all POS minus dropped, lemmatized
    tokens_all_lemma = []
    for token in doc:
        if token.pos_ in dropped_upos:
            continue
        if token.lemma_.lower() in stopwords:
            continue
        if token.is_space:
            continue
        tokens_all_lemma.append(token.lemma_.lower())
    
    #version 3: nouns only, not lemmatized
    tokens_nouns = []
    for token in doc:
        #only keep nouns and proper nouns
        if token.pos_ not in nouns_only:
            continue
        if token.text.lower() in stopwords:
            continue
        if token.is_space:
            continue
        tokens_nouns.append(token.text.lower())
    
    #version 4: nouns only, lemmatized
    tokens_nouns_lemma = []
    for token in doc:
        if token.pos_ not in nouns_only:
            continue
        if token.lemma_.lower() in stopwords:
            continue
        if token.is_space:
            continue
        tokens_nouns_lemma.append(token.lemma_.lower())
    
    #version 5: content words (nouns + verbs + adjectives), porter stemmed 
    tokens_content_stem = []
    for token in doc:
        if token.pos_ not in content_pos:
            continue
        if token.text.lower() in stopwords:
            continue
        if token.is_space:
            continue
        #apply porter stemming to the lowercase token
        stemmed_token = porter_stem(token.text.lower())
        tokens_content_stem.append(stemmed_token)
    
    #join each token list into a single string and clean it up
    text_all = clean_text(" ".join(tokens_all))
    text_all_lemma = clean_text(" ".join(tokens_all_lemma))
    text_nouns = clean_text(" ".join(tokens_nouns))
    text_nouns_lemma = clean_text(" ".join(tokens_nouns_lemma))
    text_content_stem = clean_text(" ".join(tokens_content_stem))
    
    #look up the date for this article
    article_date = date_map.get(article_id, "")
    
    #Rebuilding json: build the processed article dictionary and add it to the list
    processed_article = {
        "index": i,
        "id": article_id,
        "date": article_date,
        "text_all": text_all,
        "text_all_lemma": text_all_lemma,
        "text_nouns": text_nouns,
        "text_nouns_lemma": text_nouns_lemma,
        "text_content_stem": text_content_stem,
    }
    processed.append(processed_article)
    
    #see if not stuck
    if i % 500 == 0:
        print("processed " + str(i) + " articles out of " + str(len(processed_docs)))

#save the processed articles to a json file
save_path = "processed_data/guardian_launder_processed.json"
output_file = open(save_path, "w", encoding="utf-8")
json.dump(processed, output_file, ensure_ascii=False, indent=2)
output_file.close()

print("saved " + str(len(processed)) + " articles to " + save_path)

Split corpus into pre-2023 (1999–2022) and post-2023 (2023–2026)
Test B = all of post-2023
From pre-2023: stratified random 15% → Test A, remaining 85% → Train

In [ ]:
#Checking how many articles in total
total = len(articles)
print("total articles: " + str(total))

pre = []
test_b = []
for a in articles:
    #just the year part 
    year = a["date"][:4]
    if year <= "2022":
        pre.append(a)
    else:
        test_b.append(a)

print("pre-2023 articles (train+test_a pool): " + str(len(pre)))
print("post-2023 articles (test_b): " + str(len(test_b)))


#for stratified sampling
articles_by_year = {}
for a in pre:
    year = a["date"][:4]
    if year not in articles_by_year:
        articles_by_year[year] = []
    articles_by_year[year].append(a)
random.seed(123)

test_a = []
train = []

all_the_years = list(articles_by_year.keys())
for year in all_the_years:
    #get all articles for this year
    docs_for_this_year = articles_by_year[year]
    #shuffle the articles so we pick a random 15%
    random.shuffle(docs_for_this_year)
    
    #calculate how many articles should go into test_a for this year
    #15% of the articles for this year rounded to the nearest whole number
    total_for_this_year = len(docs_for_this_year)
    fifteen_percent = total_for_this_year * 0.15
    n_test = round(fifteen_percent)
    
    #the first n_test articles go into test_a
    articles_for_test_a = docs_for_this_year[:n_test]
    
    #the rest go into train
    articles_for_train = docs_for_this_year[n_test:]
    
    #add to the main lists
    for a in articles_for_test_a:
        test_a.append(a)
    for a in articles_for_train:
        train.append(a)

#print out the split sizes to check 
print("train: " + str(len(train)))
print("test set a: " + str(len(test_a)))
print("test set b: " + str(len(test_b)))

#check that the total adds up to the original number of articles
total_after_split = len(train) + len(test_a) + len(test_b)
print("total after split: " + str(total_after_split))

#saving train
train_save_path = "processed_data/guardian_train.json"
train_file = open(train_save_path, "w", encoding="utf-8")
json.dump(train, train_file, ensure_ascii=False, indent=2)
train_file.close()
print("saved guardian_train.json")

#saving test_a
test_a_save_path = "processed_data/guardian_test_a.json"
test_a_file = open(test_a_save_path, "w", encoding="utf-8")
json.dump(test_a, test_a_file, ensure_ascii=False, indent=2)
test_a_file.close()
print("saved guardian_test_a.json")

#saving test_b
test_b_save_path = "processed_data/guardian_test_b.json"
test_b_file = open(test_b_save_path, "w", encoding="utf-8")
json.dump(test_b, test_b_file, ensure_ascii=False, indent=2)
test_b_file.close()
print("saved guardian_test_b.json")